# 02 - Authentications Data Wrangling

Notebook ini memproses dataset `authentications.csv`.

Tabel ini berfungsi sebagai data autentikasi dan sesi pengguna. Karena tabel ini bergantung pada `users`, validasi utama diarahkan pada relasi `user_id`, kelengkapan token, dan validitas timestamp.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Dataset

In [ ]:
# memuat dataset dari folder yang sesuai dan menampilkan sampel awal data.
# Membaca file CSV ke dalam dataframe.
authentications = pd.read_csv(RAW_DIR / "authentications.csv")
users_clean = pd.read_csv(PROCESSED_DIR / "users_clean.csv")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
authentications.head()

,id,user_id,token,device_info,expires_at,created_at
0,1,1,b39fd303780b19b1942343c420cc16b723bb87f34daa35...,Firefox Linux,2026-05-10 14:00:00,2026-04-10 14:00:00
1,2,2,a515f8084a5160a9f8bd374a7312fceec1d56eac66f4ae...,Chrome Windows,2026-05-03 19:00:00,2026-04-03 19:00:00
2,3,3,1b80be6956ff3fff1043e0cff51b2d035bccc87da4e71f...,Firefox Linux,2026-05-07 03:00:00,2026-04-07 03:00:00
3,4,4,551d61b4f088322a0ba36c1a1e4b0561b6987b1d9fb18e...,Safari iOS,2026-05-10 07:00:00,2026-04-10 07:00:00
4,5,5,6545701c73c9744147dbe1cf51208000ef781d8e653f7c...,Chrome Windows,2026-05-10 13:00:00,2026-04-10 13:00:00


## 2. Assessing Data

Pemeriksaan dilakukan pada struktur data, missing value, duplicate `id`, token kosong, dan timestamp `expires_at`.

Pada tabel ini, kualitas data tidak dinilai dari pola perilaku pengguna, melainkan dari kelengkapan data sistem dan validitas relasi terhadap tabel user.

In [ ]:
# Menampilkan struktur kolom, tipe data, dan jumlah non-null.
authentications.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           300 non-null    int64
 1   user_id      300 non-null    int64
 2   token        300 non-null    str  
 3   device_info  300 non-null    str  
 4   expires_at   300 non-null    str  
 5   created_at   300 non-null    str  
dtypes: int64(2), str(4)
memory usage: 14.2 KB


In [ ]:
# Menampilkan ringkasan statistik untuk kolom numerik dan kategorikal.
authentications.describe(include='all')

,id,user_id,token,device_info,expires_at,created_at
count,300.000000,300.000000,300,300,300,300
unique,NaN,NaN,300,5,180,180
top,NaN,NaN,b39fd303780b19b1942343c420cc16b723bb87f34daa35...,Safari iOS,2026-05-10 09:00:00,2026-04-10 09:00:00
freq,NaN,NaN,1,68,5,5
mean,150.500000,150.500000,NaN,NaN,NaN,NaN
std,86.746758,86.746758,NaN,NaN,NaN,NaN
min,1.000000,1.000000,NaN,NaN,NaN,NaN
25%,75.750000,75.750000,NaN,NaN,NaN,NaN
50%,150.500000,150.500000,NaN,NaN,NaN,NaN
75%,225.250000,225.250000,NaN,NaN,NaN,NaN


In [ ]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
print("Missing value:")
# Menghitung jumlah missing value pada setiap kolom.
print(authentications.isna().sum())

# Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
print("\nDuplicate id:", authentications["id"].duplicated().sum())
# Menghitung jumlah missing value pada setiap kolom.
print("Missing token:", authentications["token"].isna().sum())
print("Missing expires_at:", authentications["expires_at"].isna().sum())

Missing value:
id             0
user_id        0
token          0
device_info    0
expires_at     0
created_at     0
dtype: int64

Duplicate id: 0
Missing token: 0
Missing expires_at: 0


## Insight:

Dataset `authentications` adalah tabel sistem. Risiko utama yang perlu dikendalikan adalah keberadaan session yang tidak memiliki user valid, token kosong, atau timestamp yang tidak dapat diproses.

Tindakan cleaning yang diperlukan adalah mengubah tipe data key, membersihkan teks pada token dan device information, memvalidasi timestamp, serta memastikan semua `user_id` mengarah ke user yang valid pada `users_clean`.

## 3. Cleaning Data

Langkah cleaning:

1. Mengubah `id` dan `user_id` ke numerik.
2. Membersihkan whitespace pada `token` dan `device_info`.
3. Mengubah `expires_at` dan `created_at` ke datetime.
4. Menghapus baris dengan key, token, atau timestamp penting yang tidak valid.
5. Mempertahankan hanya `user_id` yang ada pada `users_clean`.
6. Menghapus duplicate `id` jika ada.

In [ ]:
# membuat salinan dataframe lalu menjalankan proses cleaning sesuai hasil assessing.
authentications_clean = authentications.copy()

# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
authentications_clean["id"] = pd.to_numeric(authentications_clean["id"], errors="coerce")
authentications_clean["user_id"] = pd.to_numeric(authentications_clean["user_id"], errors="coerce")
# Membersihkan whitespace dan menstandarkan format teks.
authentications_clean["token"] = authentications_clean["token"].astype(str).str.strip()
authentications_clean["device_info"] = authentications_clean["device_info"].fillna("").astype(str).str.strip()
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
authentications_clean["expires_at"] = pd.to_datetime(authentications_clean["expires_at"], errors="coerce")
authentications_clean["created_at"] = pd.to_datetime(authentications_clean["created_at"], errors="coerce")

# Menghapus baris yang kehilangan kolom kunci atau informasi penting.
authentications_clean = authentications_clean.dropna(subset=["id", "user_id", "token", "expires_at"])

valid_user_ids = set(users_clean["id"])
authentications_clean = authentications_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    authentications_clean["user_id"].isin(valid_user_ids)
]

# Menghapus duplicate sesuai subset key yang ditentukan.
authentications_clean = authentications_clean.drop_duplicates(subset=["id"], keep="last")

authentications_clean["id"] = authentications_clean["id"].astype(int)
authentications_clean["user_id"] = authentications_clean["user_id"].astype(int)
authentications_clean["expires_at"] = authentications_clean["expires_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
authentications_clean["created_at"] = authentications_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

authentications_clean = authentications_clean[
    ["id", "user_id", "token", "device_info", "expires_at", "created_at"]
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
].sort_values("id")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
authentications_clean.head()

,id,user_id,token,device_info,expires_at,created_at
0,1,1,b39fd303780b19b1942343c420cc16b723bb87f34daa35...,Firefox Linux,2026-05-10 14:00:00,2026-04-10 14:00:00
1,2,2,a515f8084a5160a9f8bd374a7312fceec1d56eac66f4ae...,Chrome Windows,2026-05-03 19:00:00,2026-04-03 19:00:00
2,3,3,1b80be6956ff3fff1043e0cff51b2d035bccc87da4e71f...,Firefox Linux,2026-05-07 03:00:00,2026-04-07 03:00:00
3,4,4,551d61b4f088322a0ba36c1a1e4b0561b6987b1d9fb18e...,Safari iOS,2026-05-10 07:00:00,2026-04-10 07:00:00
4,5,5,6545701c73c9744147dbe1cf51208000ef781d8e653f7c...,Chrome Windows,2026-05-10 13:00:00,2026-04-10 13:00:00


## 4. Validation dan Save Output

In [ ]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
# Membuat dataframe validasi untuk mendokumentasikan hasil pengecekan kualitas data.
validation = pd.DataFrame([
    {"rule": "authentications.id unique", "passed": authentications_clean["id"].is_unique},
    {"rule": "authentications.user_id exists in users", "passed": set(authentications_clean["user_id"]).issubset(set(users_clean["id"]))},
    {"rule": "token not null", "passed": authentications_clean["token"].notna().all()},
])

validation

,rule,passed
0,authentications.id unique,True
1,authentications.user_id exists in users,True
2,token not null,True


In [ ]:
# menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
authentications_clean.to_csv(PROCESSED_DIR / "authentications_clean.csv", index=False)
validation.to_csv(REPORT_DIR / "authentications_validation.csv", index=False)

print("Saved:", PROCESSED_DIR / "authentications_clean.csv")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\authentications_clean.csv
